### Generation: Stuffing Documents


###### Setup API Key som miljøvariabel ######


In [3]:
from langchain_classic.chains.summarize.refine_prompts import prompt_template
%load_ext dotenv
%dotenv ../.env

###### Import ######

In [4]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

###### Opret Chroma Vectorstore & Embedding model ######

In [5]:
vectorstore = Chroma(persist_directory = "./chroma_db",
                     embedding_function = OpenAIEmbeddings(model='text-embedding-ada-002'))

In [6]:
len(vectorstore.get()['documents'])

22

###### Søg i Vectorstore med retriver ######


In [7]:
retriever = vectorstore.as_retriever(search_type = 'mmr',
                                     search_kwargs = {'k':3,
                                                      'lambda_mult':0.7})

###### Prompt template ######

In [8]:
TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

###### Chat model ######

In [10]:
chat = ChatOpenAI(model='gpt-4',
                  model_kwargs={'seed': 365},
                  max_tokens=250)

###### Definer brugerspørgsmål ######

In [11]:
question = "What software do data scientists use?"

###### Opbyg retieval chain med stuffing dokumenter ######

In [18]:
# Start med at lave en dictonary med questions og keys
chain = {'context': retriever,
         'question': RunnablePassthrough()} | prompt_template

In [ ]:
chain.invoke(question)

In [21]:
# Marker string som paste ind i print funn
print("\nAnswer the following question:\nWhat software do data scientists use?\n\nTo answer the question, use only the following context:\n[Document(id='c29c2d03-8d41-43d2-b663-b52b78581141', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(id='314ed61b-177c-43a7-a9ae-0740a1cf93af', metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Hadoop distributes the computational tasks on multiple computers which is basically the way to handle big data nowadays. Power BI, SaS, Qlik, and especially Tableau are top-notch examples of software designed for business intelligence visualizations'), Document(id='3d51c2d9-71e6-4ea2-851b-d8c6597a7573', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!')]\n\nAt the end of the response, specify the name of the lecture this context is taken from in the format:\nResources: *Lecture Title*\nwhere *Lecture Title* should be substituted with the title of all resource lectures.\n")


Answer the following question:
What software do data scientists use?

To answer the question, use only the following context:
[Document(id='c29c2d03-8d41-43d2-b663-b52b78581141', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(id='314ed61b-177c-43a7-a9ae-0740a1cf93af', metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in D